# Implementación de un MLPClassifier en Excel

Este tutorial muestra cómo desplegar un modelo entrenado con `MLPClassifier` de scikit-learn en una hoja de **Excel**, replicando el *forward pass* con los pesos y biases obtenidos del entrenamiento.

---

## 1. Representación general de una red neuronal

Un **MLPClassifier** en scikit-learn funciona así:

$
h^{(1)} = f\big(W^{(1)}x + b^{(1)}\big)
$

$
h^{(2)} = f\big(W^{(2)}h^{(1)} + b^{(2)}\big)
$

$
\hat{y} = g\big(W^{(L)}h^{(L-1)} + b^{(L)}\big)
$

Donde:

- $x$: vector de entrada (tus características).  
- $W^{(l)}$: matriz de pesos de la capa $l$.  
- $b^{(l)}$: bias de la capa $l$.  
- $f(\cdot)$: función de activación de las **capas ocultas** (por defecto en `MLPClassifier` es **ReLU** o **tanh**).  
- $g(\cdot)$: función de activación de la **capa de salida** (softmax para clasificación multiclase, logistic/sigmoid si es binaria).  

---

## 2. Forward pass en Excel paso a paso

### a) Primera capa oculta

Si la entrada es un vector $x = (x_1, x_2, \dots, x_n)$:

1. Multiplica $x$ por la matriz de pesos $W^{(1)}$.  
2. Suma el bias $b^{(1)}$.  
3. Aplica la activación (ejemplo con **ReLU**):  

$
h^{(1)}_j = \max\Big(0, \sum_i W^{(1)}_{ji} x_i + b^{(1)}_j \Big)
$

En Excel:

```excel
=MAX(0, SUMAPRODUCTO(rango_x ; rango_pesos) + bias)
```

---

### b) Capas ocultas siguientes

Repites lo mismo para cada capa oculta:

1. Tomas la salida de la capa anterior ($h^{(l-1)}$).  
2. Multiplicas por la matriz de pesos $W^{(l)}$.  
3. Sumas el bias $b^{(l)}$.  
4. Aplicas la función de activación.

---

### c) Capa de salida

Depende de tu problema:

- **Clasificación binaria**:  
  La salida es una **sigmoid**:  

$
\hat{y} = \frac{1}{1 + e^{-z}}
$

En Excel:

```excel
=1/(1+EXP(-z))
```

- **Clasificación multiclase**:  
  La salida es un **softmax**:  

$
\hat{y}_k = \frac{e^{z_k}}{\sum_j e^{z_j}}
$

o en su versión expandida:

$
\hat{y}_k = \frac{e^{z_k}}{e^{z_1} + e^{z_2} + \cdots + e^{z_K}}
$

Donde:

- $\hat{y}_k$: Probabilidad predicha para la clase $k$ después de aplicar softmax. Es el valor de salida para la clase $k$.
- $z_k$: Logit o salida lineal (antes de la activación) correspondiente a la clase $k$.
- $e^{z_k}$: Exponencial del logit de la clase $k$.
- $\sum_j e^{z_j}$: Suma de los exponenciales de los logits de **todas** las clases posibles. El índice $j$ recorre todas las clases ($j = 1, 2, ..., K$).
- $k$: Índice de la clase para la que se calcula la probabilidad (puede ser cualquier valor entre $1$ y $K$).
- $j$: Índice que recorre todas las clases posibles en el denominador.
- $K$: Número total de clases en el problema de clasificación.


Expresado 

En Excel (para clase $k$):

```excel
=EXP(z_k) / SUMA(EXP(rango_z))
```

In [1]:
# Import the database
import pandas as pd
penguins = pd.read_csv('/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/Datasets/palmer_penguins.csv')
penguins.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [2]:
# Preprocess the data
penguins = penguins.dropna()
penguins = penguins[penguins['sex'] != '.']
penguins.head()

# Define features and target variable
from sklearn.preprocessing import LabelEncoder
X = penguins.drop('species', axis=1)
X = X.apply(LabelEncoder().fit_transform)
y = penguins['species']

In [4]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [5]:
# Fit a MLP model
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(100, 100), activation='relu', solver='adam')
mlp.fit(X_train, y_train)

/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,"(100, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,None


In [6]:
# Make a prediction for a new penguin
new_penguin = [[1, 45, 58, 11, 22, 0]]
new_prediction = mlp.predict(new_penguin)
new_prediction

/Users/eugenio/Documents/Notebooks_ArtificialIntelligence/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


array(['Adelie'], dtype='<U9')

In [7]:
# Export the model weights and biases
weights = mlp.coefs_
biases = mlp.intercepts_

In [8]:
# Install openpyxl if not already installed
!pip install openpyxl

# Import openpyxl for Excel operations
import openpyxl

# Save weights and biases to Excel
pd.DataFrame(weights[0]).to_excel('weights_layer1.xlsx', index=False)
pd.DataFrame(weights[1]).to_excel('weights_layer2.xlsx', index=False)
pd.DataFrame(weights[2]).to_excel('weights_layer3.xlsx', index=False)
pd.DataFrame(biases[0]).to_excel('biases_layer1.xlsx', index=False)
pd.DataFrame(biases[1]).to_excel('biases_layer2.xlsx', index=False)
pd.DataFrame(biases[2]).to_excel('biases_layer3.xlsx', index=False)

### Let's go to Excel..